In [0]:
# Définir le chemin de base et le catalogue/schéma cible
base_path = "file:/Workspace/Repos/elmerahykhadija700@gmail.com/ecommerce-pipeline-v2-azure-databricks/data"
target_schema = "dbw_lab.bronze"

# S'assurer que le schéma (la base de données) bronze existe
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")

# Liste de tous les fichiers du dossier data
file_names = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]

for file in file_names:
    # Générer un nom de table propre (ex: olist_customers, olist_orders, etc.)
    table_name = file.replace(".csv", "").replace("_dataset", "")
    full_table_name = f"{target_schema}.{table_name}"
    
    print(f"Ingestion du fichier {file} vers la table {full_table_name}...")
    
    # 1. Lire le fichier CSV dans un DataFrame Spark
    df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(f"{base_path}/{file}")
    )
    
    # 2. Écrire le DataFrame en tant que table Delta dans Unity Catalog (mode overwrite ou append)
    (
        df.write
        .format("delta")
        .mode("overwrite") # Remplace la table si elle existe déjà, utilisez "append" pour ajouter
        .saveAsTable(full_table_name)
    )

print("Ingestion de la couche Bronze terminée avec succès !")

In [0]:
%pip install azure-storage-blob python-dotenv

from dotenv import load_dotenv
import os
from azure.storage.blob import BlobServiceClient

# 1. Charger les variables du fichier .env
load_dotenv()
storage_account_name = os.getenv("AZURE_STORAGE_ACCOUNT") # stolistdatalake2026
container_name = os.getenv("AZURE_CONTAINER_NAME")         # bronze
sas_token = os.getenv("AZURE_SAS_TOKEN")

# URL de connexion au service Blob avec le jeton SAS
account_url = f"https://{storage_account_name}.blob.core.windows.net?{sas_token}"
blob_service_client = BlobServiceClient(account_url=account_url)
container_client = blob_service_client.get_container_client(container_name)

# Chemin local de ton dossier de données dans le repo Databricks
base_path = "/Workspace/Repos/elmerahykhadija700@gmail.com/ecommerce-pipeline-v2-azure-databricks/data"

file_names = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]

for file in file_names:
    local_file_path = os.path.join(base_path, file)
    blob_name = f"raw_csv/{file}" # Nom du fichier dans ton conteneur Azure
    
    print(f"Téléversement de {file} vers le conteneur Azure ({container_name})...")
    
    blob_client = container_client.get_blob_client(blob_name)
    
    with open(local_file_path, "rb") as data:
        blob_client.upload_blob(data, overwrite=True)

print("🚀 Tous les fichiers de la couche Bronze ont été téléversés avec succès dans le cloud Azure !")